In [1]:
import pandas as pd
df = pd.read_csv("uybor.csv")

In [2]:
x = [9, 3]
w = [0.4, -0.6]
b = -1

result = w[0] * x[0] + w[1] * x[1] + b
print(result)

0.8000000000000003


In [3]:
natija = 1 if result >= 0 else 0
print(natija)

1


In [4]:
import numpy as np
def sigmoid(z):
  return 1/(1+np.exp(-z))

print(sigmoid(result))

0.6899744811276125


In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

In [6]:
df = pd.read_csv('uybor.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Unnamed: 0.1        518 non-null    int64
 1   Unnamed: 0          518 non-null    int64
 2   Title               518 non-null    str  
 3   Square              518 non-null    str  
 4   Price               518 non-null    str  
 5   Room                518 non-null    str  
 6   Floor               518 non-null    int64
 7   Building floor      518 non-null    int64
 8   Renovation          518 non-null    str  
 9   Material            518 non-null    str  
 10  Address             518 non-null    str  
 11  Lift                518 non-null    int64
 12  Bathroom            518 non-null    int64
 13  Washing machine     518 non-null    int64
 14  TV                  518 non-null    int64
 15  Microwave oven      518 non-null    int64
 16  Playground          518 non-null    int64
 17  Furnitur

In [7]:
df.drop(['Title','Unnamed: 0.1','Unnamed: 0'],axis=1, inplace=True)

In [8]:
def price(value):
    value = value.replace(' у.е.', '').replace(' ','')
    return float(value)
df['Price'] = df['Price'].apply(price)

In [9]:
def size(value):
    value = value.replace(' м²', '').replace('м²', '').replace(' ', '')
    return float(value)

df['Square'] = df['Square'].apply(size)

In [10]:
def room(value):
    value = value.replace('+', '').strip()
    return int(value)
df['Room'] = df['Room'].apply(room)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 518 entries, 0 to 517
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Square              518 non-null    float64
 1   Price               518 non-null    float64
 2   Room                518 non-null    int64  
 3   Floor               518 non-null    int64  
 4   Building floor      518 non-null    int64  
 5   Renovation          518 non-null    str    
 6   Material            518 non-null    str    
 7   Address             518 non-null    str    
 8   Lift                518 non-null    int64  
 9   Bathroom            518 non-null    int64  
 10  Washing machine     518 non-null    int64  
 11  TV                  518 non-null    int64  
 12  Microwave oven      518 non-null    int64  
 13  Playground          518 non-null    int64  
 14  Furniture           518 non-null    int64  
 15  Refrigerator        518 non-null    int64  
 16  Air conditioner    

In [12]:
x = df.drop('Price', axis=1)
y = df['Price']

In [13]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [14]:
numerical_features = x_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x_train.select_dtypes(include=['object']).columns

C:\Users\Admin\AppData\Local\Temp\ipykernel_25104\2571878621.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x_train.select_dtypes(include=['object']).columns


In [15]:
print(numerical_features)
print(categorical_features)

Index(['Square', 'Room', 'Floor', 'Building floor', 'Lift', 'Bathroom',
       'Washing machine', 'TV', 'Microwave oven', 'Playground', 'Furniture',
       'Refrigerator', 'Air conditioner', 'Wifi', 'Cable', 'Security',
       'Video surveillance', 'Parking space'],
      dtype='str')
Index(['Renovation', 'Material', 'Address'], dtype='str')


In [16]:
sc = StandardScaler()
oe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
num_transformer = Pipeline(
    steps=[
      ('imputer', SimpleImputer(strategy='mean')),
      ('scaler', sc)
     ]
    )

cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', oe)
    ]
    )

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features)
    ]
)

In [18]:
x_train_preprocessed = preprocessor.fit_transform(x_train)
y_train_preprocessed = np.log1p(y_train)

In [19]:
x_test_preprocessed = preprocessor.transform(x_test)
y_test_preprocessed = np.log1p(y_test.values)

d:\Programming\Coding UStudy\Atificial Intelligence\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [20]:
x_train_tensor = torch.FloatTensor(x_train_preprocessed)
x_test_tensor = torch.FloatTensor(x_test_preprocessed)
y_train_tensor = torch.FloatTensor(y_train_preprocessed.values).reshape(-1, 1)
y_test_tensor = torch.FloatTensor(y_test_preprocessed).reshape(-1, 1)

C:\Users\Admin\AppData\Local\Temp\ipykernel_25104\3062408597.py:3: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.FloatTensor(y_train_preprocessed.values).reshape(-1, 1)


In [21]:

print("\n✅ Tensors created from PREPROCESSED data!")
print(f"x_train_tensor: {x_train_tensor.shape}")
print(f"x_test_tensor: {x_test_tensor.shape}")
print(f"y_train_tensor: {y_train_tensor.shape}")
print(f"y_test_tensor: {y_test_tensor.shape}")


✅ Tensors created from PREPROCESSED data!
x_train_tensor: torch.Size([414, 441])
x_test_tensor: torch.Size([104, 441])
y_train_tensor: torch.Size([414, 1])
y_test_tensor: torch.Size([104, 1])


In [22]:
class HomePrices(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.layer_1 = nn.Linear(input_features, 32)
        self.layer_2 = nn.Linear(32, 16)
        self.layer_3 = nn.Linear(16, 8)
        self.layer_4 = nn.Linear(8, 4)
        self.layer_5 = nn.Linear(4, 2)
        self.layer_6 = nn.Linear(2, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.layer_1(x))
        x = self.relu(self.layer_2(x))
        x = self.dropout(x)
        x = self.relu(self.layer_3(x))
        x = self.dropout(x)
        x = self.relu(self.layer_4(x))
        x = self.dropout(x)
        x = self.relu(self.layer_5(x))
        x = self.dropout(x)
        x = self.layer_6(x)

        return x


In [23]:
input_size = x_train_tensor.shape[1]
model = HomePrices(input_size)

In [24]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [25]:
num_epoch = 200
batch_size = 64
train_losses = []
test_losses = []

In [26]:
for i in range(num_epoch):
    model.train()
    epoch_loss = 0.0

    for j in range(0, len(x_train_tensor), batch_size):
        batch_x = x_train_tensor[j:j+batch_size]
        batch_y = y_train_tensor[j:j+batch_size]

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

    avg_train_loss = epoch_loss / len(x_train_tensor)
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        test_outputs = model(x_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_losses.append(test_loss.item())

    if (i + 1) % 20 == 0:
        print(f"Epoch {i + 1} / {num_epoch} Train loss: {avg_train_loss} Test loss: {test_loss.item()}")

model.eval()
with torch.no_grad():
    train_outputs = model(x_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_mse = mean_squared_error(y_train_tensor.numpy(), train_outputs.numpy())
    train_rmse = np.sqrt(train_mse)

    test_outputs = model(x_test_tensor)
    test_loss = criterion(test_outputs, y_test_tensor)
    test_mse = mean_squared_error(y_test_tensor.numpy(), test_outputs.numpy())
    test_rmse = np.sqrt(test_mse)

print("training loss: ", train_loss.item())
print("training MSE: ", train_mse)
print("training RMSE: ", train_rmse)
print("test loss: ", test_loss.item())
print("test MSE: ", test_mse)
print("test RMSE: ", test_rmse)


Epoch 20 / 200 Train loss: 121.55252731249529 Test loss: 121.13529205322266
Epoch 40 / 200 Train loss: 120.03568474221345 Test loss: 119.56900024414062
Epoch 60 / 200 Train loss: 118.38793491971666 Test loss: 117.66963958740234
Epoch 80 / 200 Train loss: 117.45258615097562 Test loss: 116.56961059570312
Epoch 100 / 200 Train loss: 117.04375067079701 Test loss: 115.91399383544922
Epoch 120 / 200 Train loss: 116.68490626385822 Test loss: 115.52660369873047
Epoch 140 / 200 Train loss: 116.20042622146975 Test loss: 115.16167449951172
Epoch 160 / 200 Train loss: 115.82205461879859 Test loss: 114.840087890625
Epoch 180 / 200 Train loss: 115.44038940282259 Test loss: 114.54153442382812
Epoch 200 / 200 Train loss: 115.13415656343174 Test loss: 114.24406433105469
training loss:  114.69280242919922
training MSE:  114.69280242919922
training RMSE:  10.709472556069192
test loss:  114.24406433105469
test MSE:  114.24405670166016
test RMSE:  10.688501143830232
